## Notes

In [20]:
# import os
# import pandas as pd
# import numpy as np

# os.listdir(path)                              # List files in a folder
# [f for f in os.listdir(path) if f.endswith(".csv_files")]  # List csv_files files only

# pd.read_csv_files(path)                             # Read csv_files file
# pd.read_csv_files(path, encoding="cp1252")          # Read csv_files with specified encoding
# pd.read_csv_files(path, usecols=["A", "B"])         # Read selected columns only

# df.dropna()                                   # Remove rows containing missing values
# df.dropna(subset=["Time", "Primary"])         # Remove rows missing selected values
# df.fillna(0)                                  # Replace missing values with 0
# df.isna()                                     # Check missing values
# df.isna().sum()                               # Count missing values per column

# pd.concat([df1, df2])                         # Combine DataFrames vertically
# pd.concat([df1, df2], ignore_index=True)      # Combine and create a new index
# pd.concat([df1, df2], axis=1)                 # Combine DataFrames horizontally

# pd.DataFrame({...})                           # Create a DataFrame
# df.copy()                                     # Create an independent copy

# df.shape                                      # Get (rows, columns)
# len(df)                                       # Get number of rows
# df.size                                       # Get total number of values
# df.head()                                     # Preview first 5 rows
# df.head(10)                                   # Preview first 10 rows
# df.tail()                                     # Preview last 5 rows
# df.sample(5)                                  # Show 5 random rows
# df.info()                                     # Show structure and data types
# df.describe()                                 # Show numerical summary statistics
# df.dtypes                                     # Show data type of every column

# df.columns                                    # View column names
# df.columns.tolist()                           # Convert column names to a list
# df.columns = df.columns.str.strip()           # Remove spaces around column names
# df.rename(columns={"Old": "New"})             # Rename one column
# df.rename(columns={"I1": "Primary", "I2": "Secondary"})  # Rename multiple columns

# df["Col"]                                     # Select one column as a Series
# df[["Col1", "Col2"]]                          # Select multiple columns
# df.loc[:, ["Col1", "Col2"]]                   # Select columns by name
# df.iloc[:, 0]                                 # Select first column by position
# df.iloc[0]                                    # Select first row
# df.iloc[0:5]                                  # Select first 5 rows

# df[df["Trial"] == 1]                          # Filter rows using one condition
# df[(df["Material"] == "Metal") & (df["Trial"] == 1)]  # Filter using multiple conditions
# df[df["Col"].isin(["A", "B"])]                # Keep selected values
# df.query("Trial == 1")                        # Filter using query syntax

# df.sort_values("Col")                         # Sort by one column
# df.sort_values(["Trial", "Time"])             # Sort by multiple columns
# df.sort_values("Col", ascending=False)        # Sort in descending order

# df.reset_index(drop=True)                     # Reset row index
# df.set_index("Time")                          # Set a column as the index

# df["NewCol"] = values                                 # Add or replace a column
# df["Difference"] = df["Primary"] - df["Secondary"]    # Create calculated column
# df.insert(1, "NewCol", values)                        # Insert column at a position
# df.drop(columns=["Col"])                              # Remove a column
# df.drop(index=[0, 1])                                 # Remove selected rows

# df["Col"].unique()                            # Get unique values
# df["Col"].nunique()                           # Count unique values
# df["Col"].value_counts()                      # Count occurrences of each value

# df.groupby("Col")                                                 # Group rows by one column
# df.groupby(["Material", "Trial"])                                 # Group using multiple columns
# df.groupby("Material")["Primary"].mean()                          # Mean Primary for each material
# df.groupby("Trial").agg({"Primary": "mean", "Secondary": "max"})  # Multiple summaries

# df["Col"].to_numpy()                          # Convert Series to NumPy array
# df["Col"].to_numpy(dtype=float)               # Convert to float NumPy array
# df["Col"].tolist()                            # Convert Series to Python list

# df["Col"].astype(float)                       # Convert column to float
# df["Col"].astype(str)                         # Convert column to string
# pd.to_numeric(df["Col"], errors="coerce")     # Convert to numeric; invalid values become NaN

# df["Col"].min()                               # Minimum value
# df["Col"].max()                               # Maximum value
# df["Col"].mean()                              # Mean
# df["Col"].median()                            # Median
# df["Col"].std()                               # Standard deviation
# df["Col"].sum()                               # Sum
# df["Col"].idxmax()                            # Index of maximum value
# df["Col"].idxmin()                            # Index of minimum value

# df.duplicated()                               # Check duplicate rows
# df.drop_duplicates()                          # Remove duplicate rows

# df.replace([np.inf, -np.inf], np.nan)         # Replace infinities with NaN
# df.loc[mask]                                  # Select rows using a Boolean mask

# df.to_csv_files("output.csv_files", index=False)          # Save DataFrame to csv_files
# df.to_excel("output.xlsx", index=False)       # Save DataFrame to Excel

# pd.set_option("display.max_columns", None)     # Display all columns
# pd.set_option("display.max_rows", None)        # Display all rows
# pd.reset_option("display.max_columns")         # Reset column display setting
# pd.reset_option("display.max_rows")            # Reset row display setting

# Sensor Data Exploration
This notebook provides a concise workflow for exploring multiple sensor datasets with pandas. It covers:

- Loading sensor files (CSV/JSON) and combining them into a single DataFrame.
- Parsing and aligning timestamps, setting a datetime index.
- Cleaning steps: handling missing values, duplicates, outliers, and unit/column harmonization.
- Quick descriptive statistics, group summaries by sensor/type, and checks for data quality.
- Time-series operations: resampling, rolling aggregates, and interpolation to common frequencies.
- Visualizations: time series plots, distributions, correlation matrices, and simple anomaly detection.
- Exporting cleaned datasets and prepared features for downstream modeling or reporting.

Expected inputs: one or more files containing timestamp, sensor_id (or similar), and measurement columns.  
Outputs: a cleaned, merged DataFrame, summary tables, and plots to guide further analysis.

In [21]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator, ScalarFormatter

from scipy.signal import savgol_filter#, correlate, correlation_lags

from tsfresh import extract_features

In [22]:
FILE_TITLE = "30C"
DATA_PATH = f"../data/02_preprocessed/{FILE_TITLE}/"

In [23]:
DATA = pd.DataFrame()
file_list = os.listdir(DATA_PATH)

for f in file_list:
    if ".csv" in f:
        trial_file = pd.read_csv(f"{DATA_PATH}{f}").dropna()
        DATA = pd.concat([DATA , trial_file] , ignore_index = True)

MATERIALS = list(DATA['Sample'].unique())

MATERIAL_GROUPS = {
'bismuth': 'metal',
'nickel': 'metal',
'titanium': 'metal',
'iron': 'metal',
'aluminum': 'metal',
'copper': 'metal',

'cement': 'ceramic',
'gypsum': 'ceramic',
'carbon': 'ceramic', 'graphite': 'ceramic',

'cork_coarse': 'composite',
'cork_fine': 'composite',
'wood': 'composite',
'abrasive': 'composite',

'pdms': 'polymer',

'ps_foam': 'foam',
'pu_foam': 'foam'}

DATA['Label']=DATA['Sample'].apply(lambda x : MATERIAL_GROUPS[x])

# Determination of elbow `time`, `value` , and `index`

**Function name:**
- find_first_elbow()

**Returned values:**
- time[elbow_position]
- signal[elbow_position]
- indices[elbow_position]
- time
- smooth_signal

In [24]:
def find_first_elbow(
    subset,
    time_col="Time",
    signal_col="Primary",
    index_col="index",
    smooth_window=15,
    polyorder=2,
    threshold_frac=0.30,
    direction="drop",
    skip_index=5,
):
    df = subset.sort_values(time_col, kind="stable").reset_index(drop=True)

    indices = (
        df[index_col].to_numpy()
        if index_col in df.columns
        else np.arange(len(df))
    )

    df = df.iloc[skip_index:]
    indices = indices[skip_index:]

    time = df[time_col].to_numpy(dtype=float)
    signal = df[signal_col].to_numpy(dtype=float)

    valid = np.isfinite(time) & np.isfinite(signal)
    time = time[valid]
    signal = signal[valid]
    indices = indices[valid]

    n = len(signal)

    if n < 3:
        return np.nan, np.nan, None, time, signal

    window = min(smooth_window, n)

    if window % 2 == 0:
        window -= 1

    if window <= polyorder:
        window = polyorder + 1

        if window % 2 == 0:
            window += 1

    if window > n:
        return np.nan, np.nan, None, time, signal

    smooth_signal = savgol_filter(
        signal,
        window_length=window,
        polyorder=polyorder,
    )

    derivative = np.gradient(smooth_signal, time)

    if direction == "drop":
        strongest_idx = np.argmin(derivative)
        active = derivative < threshold_frac * derivative[strongest_idx]

    elif direction == "rise":
        strongest_idx = np.argmax(derivative)
        active = derivative > threshold_frac * derivative[strongest_idx]

    else:
        raise ValueError("direction must be 'drop' or 'rise'")

    elbow_position = 0

    for position in range(strongest_idx, -1, -1):
        if not active[position]:
            elbow_position = position + 1
            break

    return (
        time[elbow_position],
        signal[elbow_position],
        indices[elbow_position],
        time,
        smooth_signal,
    )

## Storage of elbow `time`, `value` , and `index` into df.DATA

In [25]:
ELBOW_IDX_DICT = {}
DATA["Elbow time"] = np.nan
DATA["Elbow value"] = np.nan
DATA["Elbow index"] = np.nan

for material in MATERIALS:
    current_material = DATA[DATA['Sample'] == material]
    current_material = current_material.sort_values(["Trial", "Time"])

    trial_numbers = sorted(current_material["Trial"].unique())

    ELBOW_IDX_DICT[material] = {}

    for trial in trial_numbers:
        trial = int(trial)
        subset = current_material[current_material["Trial"] == trial]

        (
            ELBOW_TIME,
            ELBOW_VALUE,
            ELBOW_IDX,
            SMOOTH_TIME,
            SMOOTH_PRIMARY,
        ) = find_first_elbow(subset)

        ELBOW_IDX_DICT[material][trial] = int(ELBOW_IDX)

        row_condition = ((DATA["Sample"] == material) & (DATA["Trial"] == trial))

        DATA.loc[row_condition, "Elbow time"]   = ELBOW_TIME
        DATA.loc[row_condition, "Elbow value"]  = ELBOW_VALUE
        DATA.loc[row_condition, "Elbow index"]  = ELBOW_IDX

In [26]:
DATA

,index,Sample,Trial,k,Mass,Volume,rho,cp,Time,Primary,Secondary,Label,Elbow time,Elbow value,Elbow index


## Visual inspection of elbow

In [27]:
for material in MATERIALS:
    current_material = DATA[DATA["Sample"] == material]
    trial_numbers = sorted(current_material["Trial"].unique())

    n_cols = min(3, len(trial_numbers))
    n_rows = math.ceil(len(trial_numbers) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trial_numbers):
        subset = (
            current_material[current_material["Trial"].astype(int) == trial]
            .sort_values("Time")
            .copy()
        )

        elbow_time = subset["Elbow time"].iloc[0]
        elbow_value = subset["Elbow value"].iloc[0]
        elbow_idx = subset["Elbow index"].iloc[0]

        ax.plot(
            subset["Time"],
            subset["Primary"],
            label="Primary",
        )

        ax.plot(
            subset["Time"],
            subset["Secondary"],
            label="Secondary",
        )

        if np.isfinite(elbow_time):
            ax.axvline(
                elbow_time,
                linestyle=":",
                label="Elbow",
            )

            ax.scatter(
                elbow_time,
                elbow_value,
                zorder=5,
            )

        ax.set(
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
            xlabel="Time",
            ylabel="Value",
        )

        ax.legend(fontsize=8)

        print(
            f"{material} - Trial {trial} - "
            f"elbow index: {elbow_idx}, elbow time: {elbow_time:.3f}"
        )

    for ax in axes[len(trial_numbers):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} elbow visual inspection",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Elbow Position Normalization

In [28]:
DATA

,index,Sample,Trial,k,Mass,Volume,rho,cp,Time,Primary,Secondary,Label,Elbow time,Elbow value,Elbow index


In [29]:
DATA["Time from elbow"] = DATA["Time"] - DATA["Elbow time"]
DATA["Primary aligned"] = DATA["Primary"] - DATA["Elbow value"]
DATA["Secondary aligned"] = DATA["Secondary"] - DATA["Elbow value"]
DATA = DATA[(DATA['Time from elbow'] >= 0) & (DATA['Time from elbow'] < 5.1)]


#  Plots of sensor data

## Plots of each material at each trial

In [30]:
for material in MATERIALS:
    material_data = DATA[
        (DATA["Sample"] == material)
        & (DATA["Time from elbow"] >= 0)
        & (DATA["Time from elbow"] <= 5)
    ].copy()

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    n_cols = min(3, len(trials))
    n_rows = math.ceil(len(trials) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trials):
        subset = (
            material_data[material_data["Trial"].astype(int) == trial]
            .sort_values("Time from elbow")
            .copy()
        )

        if subset.empty:
            ax.axis("off")
            continue

        ax.plot(
            subset["Time from elbow"],
            subset["Primary aligned"],
            label="Primary aligned",
        )

        ax.plot(
            subset["Time from elbow"],
            subset["Secondary aligned"],
            label="Secondary aligned",
        )

        ax.axvline(
            0,
            linestyle=":",
            label="Elbow",
        )

        ax.scatter(
            0,
            0,
            zorder=5,
        )

        ax.set_xlim(0, 5)

        ax.set(
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
            xlabel="Time from elbow (s)",
            ylabel="Aligned current (A)",
        )

        ax.legend(fontsize=8)

    for ax in axes[len(trials):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - Position-normalized response",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Plot of each material across all trials

In [31]:
for material in MATERIALS:
    material_data = DATA[
        DATA["Sample"] == material
    ].copy()

    if material_data.empty:
        print(f"No post-elbow data for {material}")
        continue

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 6),
        sharey=False,
        squeeze=False,
    )

    ax_both = axes.ravel()[0]
    ax_primary = axes.ravel()[1]

    cmap = plt.get_cmap("tab10", len(trials))

    for trial_idx, trial in enumerate(trials):
        subset = (
            material_data[material_data["Trial"].astype(int) == trial]
            .sort_values("Time from elbow")
            .copy()
        )

        trial_color = cmap(trial_idx)

        # Left plot: aligned Primary + aligned Secondary
        ax_both.plot(
            subset["Time from elbow"],
            subset["Primary aligned"],
            color=trial_color,
            linestyle="-",
            linewidth=2,
        )

        ax_both.plot(
            subset["Time from elbow"],
            subset["Secondary aligned"],
            color=trial_color,
            linestyle="--",
            linewidth=2,
        )

        # Right plot: aligned Primary only
        ax_primary.plot(
            subset["Time from elbow"],
            subset["Primary aligned"],
            color=trial_color,
            linestyle="-",
            linewidth=2,
        )

    for ax in [ax_both, ax_primary]:
        ax.axvline(
            0,
            linestyle=":",
            color="black",
            linewidth=2,
        )

        ax.axhline(
            0,
            linestyle=":",
            color="gray",
            linewidth=1,
        )

        ax.set_xlim(0, 5)

        ax.grid(
            linestyle=":",
            alpha=0.5,
        )

    ax_both.set(
        title="Primary and Secondary aligned signals",
        xlabel="Time from elbow (s)",
        ylabel="Aligned current (A)",
    )

    ax_primary.set(
        title="Primary aligned signal only",
        xlabel="Time from elbow (s)",
        ylabel="Aligned primary current (A)",
    )

    trial_handles = [
        Line2D(
            [0],
            [0],
            color=cmap(i),
            linestyle="-",
            linewidth=2,
            label=f"Trial {trial}",
        )
        for i, trial in enumerate(trials)
    ]

    sensor_handles = [
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="-",
            linewidth=2,
            label="Primary aligned",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle="--",
            linewidth=2,
            label="Secondary aligned",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            linestyle=":",
            linewidth=2,
            label="Elbow",
        ),
    ]

    legend1 = ax_both.legend(
        handles=trial_handles,
        title="Trial",
        loc="upper right",
        fontsize=8,
    )

    ax_both.add_artist(legend1)

    ax_both.legend(
        handles=sensor_handles,
        title="Signal",
        loc="lower right",
        fontsize=8,
    )

    ax_primary.legend(
        handles=trial_handles,
        title="Trial",
        loc="upper right",
        fontsize=8,
    )

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - All trials position-normalized",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Plot of all materials per trial

In [32]:
trials = sorted(
    DATA["Trial"]
    .dropna()
    .astype(int)
    .unique()
)

materials = [
    material
    for material in MATERIALS
    if material in DATA["Sample"].unique()
]

cmap = plt.get_cmap("tab20", len(materials))

for trial in trials:
    trial_data = (
        DATA[DATA["Trial"].astype(int) == trial]
        .copy()
        .sort_values(["Sample", "Time from elbow"])
    )

    fig, ax = plt.subplots(figsize=(12, 7))

    for material_idx, material in enumerate(materials):
        subset = (
            trial_data[trial_data["Sample"] == material]
            .sort_values("Time from elbow")
            .copy()
        )

        if subset.empty:
            continue

        material_color = cmap(material_idx)

        ax.plot(
            subset["Time from elbow"],
            subset["Primary aligned"],
            color=material_color,
            linewidth=2,
            label=material.replace("_", " ").title(),
        )

    ax.axvline(
        0,
        linestyle=":",
        color="black",
        linewidth=2,
        label="Elbow",
    )

    ax.axhline(
        0,
        linestyle=":",
        color="gray",
        linewidth=1,
    )

    ax.set_xlim(0, 5)

    ax.set(
        title=f"Trial {trial} - All materials",
        xlabel="Time from elbow (s)",
        ylabel="Aligned primary current (A)",
    )

    ax.grid(linestyle=":", alpha=0.5)

    ax.legend(
        title="Material",
        fontsize=8,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    plt.tight_layout()
    plt.show()

In [33]:
DATA[['Sample', 'Trial']].value_counts()

Series([], Name: count, dtype: int64)

## Plot of `primary`-`secondary` signals

In [34]:
DATA["diff aligned"] = (
    DATA["Secondary aligned"] - DATA["Primary aligned"]
)

DATA["abs diff aligned"] = DATA["diff aligned"].abs()

for material in MATERIALS:
    material_data = DATA[
        (DATA["Sample"] == material)
        & (DATA["Time from elbow"] >= 0)
        & (DATA["Time from elbow"] <= 5)
    ].copy()

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    n_cols = min(3, len(trials))
    n_rows = math.ceil(len(trials) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trials):
        subset = (
            material_data[material_data["Trial"].astype(int) == trial]
            .sort_values("Time from elbow")
            .copy()
        )

        ax.plot(
            subset["Time from elbow"],
            subset["diff aligned"],
            label="Secondary aligned - Primary aligned",
        )

        #ax.axvline(
        #    0,
        #    linestyle=":",
        #    label="Elbow",
        #)

        #ax.axhline(
        #    0,
        #    linestyle=":",
        #    linewidth=1,
        #)

        #ax.set_xlim(0, 5)

        ax.set(
            xlabel="Time from elbow (s)",
            ylabel="Aligned difference current (A)",
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
        )

        ax.legend(fontsize=8)

    for ax in axes[len(trials):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - Difference signal",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Plot of first-order derivatives

In [35]:
# ------------------------------------------------------------
# Optional smoothing before derivative
# ------------------------------------------------------------
def smooth_signal(y, window=11, polyorder=2):
    y = np.asarray(y, dtype=float)

    if len(y) < 5:
        return y

    window = min(window, len(y))

    # Savitzky-Golay window must be odd
    if window % 2 == 0:
        window -= 1

    min_window = polyorder + 2
    if min_window % 2 == 0:
        min_window += 1

    if window < min_window:
        return y

    return savgol_filter(
        y,
        window_length=window,
        polyorder=polyorder,
        mode="interp",
    )


# ------------------------------------------------------------
# Plot d(signal)/dt for Primary and Secondary
# ------------------------------------------------------------
for material in MATERIALS:

    material_data = DATA[
        (DATA["Sample"] == material)
        & (DATA["Time from elbow"] >= 0)
        & (DATA["Time from elbow"] <= 5)
    ].copy()

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    if len(trials) == 0:
        continue

    n_cols = min(3, len(trials))
    n_rows = math.ceil(len(trials) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trials):

        subset = (
            material_data[
                material_data["Trial"].astype(int) == trial
            ]
            .sort_values("Time from elbow")
            .copy()
        )

        time = subset["Time from elbow"].to_numpy(dtype=float)
        primary = subset["Primary aligned"].to_numpy(dtype=float)
        secondary = subset["Secondary aligned"].to_numpy(dtype=float)

        # Skip invalid or too-short trials
        if len(time) < 3:
            ax.axis("off")
            continue

        # Smooth before derivative to reduce noise amplification
        primary_smooth = smooth_signal(
            primary,
            window=11,
            polyorder=2,
        )

        secondary_smooth = smooth_signal(
            secondary,
            window=11,
            polyorder=2,
        )

        # d(signal)/dt
        primary_dsignal_dt = np.gradient(
            primary_smooth,
            time,
        )

        secondary_dsignal_dt = np.gradient(
            secondary_smooth,
            time,
        )

        ax.plot(
            time,
            primary_dsignal_dt,
            label="Primary d(signal)/dt",
            linewidth=2,
        )

        ax.plot(
            time,
            secondary_dsignal_dt,
            label="Secondary d(signal)/dt",
            linewidth=2,
            linestyle="--",
        )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=1,
        )

        ax.set(
            xlabel="Time from elbow (s)",
            ylabel="d(signal)/dt (A/s)",
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
        )

        ax.grid(
            linestyle=":",
            alpha=0.5,
        )

        ax.legend(fontsize=8)

    for ax in axes[len(trials):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - Primary and Secondary Rate Profiles",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Plot of differences between `primary` and `secondary` first-order derivatives

In [36]:
def smooth_signal(y, window=11, polyorder=2):
    y = np.asarray(y, dtype=float)

    if len(y) < 5:
        return y

    window = min(window, len(y))

    if window % 2 == 0:
        window -= 1

    min_window = polyorder + 2
    if min_window % 2 == 0:
        min_window += 1

    if window < min_window:
        return y

    return savgol_filter(
        y,
        window_length=window,
        polyorder=polyorder,
        mode="interp",
    )


for material in MATERIALS:

    material_data = DATA[
        (DATA["Sample"] == material)
        & (DATA["Time from elbow"] >= 0)
        & (DATA["Time from elbow"] <= 5)
    ].copy()

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    if len(trials) == 0:
        continue

    n_cols = min(3, len(trials))
    n_rows = math.ceil(len(trials) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trials):

        subset = (
            material_data[
                material_data["Trial"].astype(int) == trial
            ]
            .sort_values("Time from elbow")
            .copy()
        )

        time = subset["Time from elbow"].to_numpy(dtype=float)
        primary = subset["Primary aligned"].to_numpy(dtype=float)
        secondary = subset["Secondary aligned"].to_numpy(dtype=float)

        if len(time) < 3:
            ax.axis("off")
            continue

        primary_smooth = smooth_signal(
            primary,
            window=11,
            polyorder=2,
        )

        secondary_smooth = smooth_signal(
            secondary,
            window=11,
            polyorder=2,
        )

        primary_dsignal_dt = np.gradient(
            primary_smooth,
            time,
        )

        secondary_dsignal_dt = np.gradient(
            secondary_smooth,
            time,
        )

        derivative_difference = (
            secondary_dsignal_dt
            - primary_dsignal_dt
        )

        ax.plot(
            time,
            derivative_difference,
            linewidth=2,
            label="Secondary rate - Primary rate",
        )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=1,
        )

        ax.set(
            xlabel="Time from elbow (s)",
            ylabel="Difference in d(signal)/dt (A/s)",
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
        )

        ax.grid(
            linestyle=":",
            alpha=0.5,
        )

        ax.legend(fontsize=8)

    for ax in axes[len(trials):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - Derivative Difference",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Plot of second-order derivatives

In [37]:
# ------------------------------------------------------------
# Optional smoothing before derivative
# ------------------------------------------------------------
def smooth_signal(y, window=11, polyorder=2):
    y = np.asarray(y, dtype=float)

    if len(y) < 5:
        return y

    window = min(window, len(y))

    # Savitzky-Golay window must be odd
    if window % 2 == 0:
        window -= 1

    min_window = polyorder + 2
    if min_window % 2 == 0:
        min_window += 1

    if window < min_window:
        return y

    return savgol_filter(
        y,
        window_length=window,
        polyorder=polyorder,
        mode="interp",
    )


# ------------------------------------------------------------
# Plot second derivative d²(signal)/dt² for Primary and Secondary
# ------------------------------------------------------------
for material in MATERIALS:

    material_data = DATA[
        (DATA["Sample"] == material)
        & (DATA["Time from elbow"] >= 0)
        & (DATA["Time from elbow"] <= 5)
    ].copy()

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    if len(trials) == 0:
        continue

    n_cols = min(3, len(trials))
    n_rows = math.ceil(len(trials) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trials):

        subset = (
            material_data[
                material_data["Trial"].astype(int) == trial
            ]
            .sort_values("Time from elbow")
            .copy()
        )

        time = subset["Time from elbow"].to_numpy(dtype=float)
        primary = subset["Primary aligned"].to_numpy(dtype=float)
        secondary = subset["Secondary aligned"].to_numpy(dtype=float)

        # Need enough points for two derivatives
        if len(time) < 5:
            ax.axis("off")
            continue

        # Smooth before derivative to reduce noise amplification
        primary_smooth = smooth_signal(
            primary,
            window=11,
            polyorder=2,
        )

        secondary_smooth = smooth_signal(
            secondary,
            window=11,
            polyorder=2,
        )

        # First derivatives: d(signal)/dt
        primary_dsignal_dt = np.gradient(
            primary_smooth,
            time,
        )

        secondary_dsignal_dt = np.gradient(
            secondary_smooth,
            time,
        )

        # Second derivatives: d²(signal)/dt²
        primary_d2signal_dt2 = np.gradient(
            primary_dsignal_dt,
            time,
        )

        secondary_d2signal_dt2 = np.gradient(
            secondary_dsignal_dt,
            time,
        )

        ax.plot(
            time,
            primary_d2signal_dt2,
            label="Primary d²(signal)/dt²",
            linewidth=2,
        )

        ax.plot(
            time,
            secondary_d2signal_dt2,
            label="Secondary d²(signal)/dt²",
            linewidth=2,
            linestyle="--",
        )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=1,
        )

        ax.set(
            xlabel="Time from elbow (s)",
            ylabel="d²(signal)/dt² (A/s²)",
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
        )

        ax.grid(
            linestyle=":",
            alpha=0.5,
        )

        ax.legend(fontsize=8)

    for ax in axes[len(trials):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - Primary and Secondary Second-Derivative Profiles",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Plot of differences between `primary` and `secondary` second-order derivatives

In [38]:
# ------------------------------------------------------------
# Optional smoothing before derivative
# ------------------------------------------------------------
def smooth_signal(y, window=11, polyorder=2):
    y = np.asarray(y, dtype=float)

    if len(y) < 5:
        return y

    window = min(window, len(y))

    # Savitzky-Golay window must be odd
    if window % 2 == 0:
        window -= 1

    min_window = polyorder + 2
    if min_window % 2 == 0:
        min_window += 1

    if window < min_window:
        return y

    return savgol_filter(
        y,
        window_length=window,
        polyorder=polyorder,
        mode="interp",
    )


# ------------------------------------------------------------
# Plot difference between second derivatives:
# d²(Secondary)/dt² - d²(Primary)/dt²
# ------------------------------------------------------------
for material in MATERIALS:

    material_data = DATA[
        (DATA["Sample"] == material)
        & (DATA["Time from elbow"] >= 0)
        & (DATA["Time from elbow"] <= 5)
    ].copy()

    trials = sorted(
        material_data["Trial"]
        .dropna()
        .astype(int)
        .unique()
    )

    if len(trials) == 0:
        continue

    n_cols = min(3, len(trials))
    n_rows = math.ceil(len(trials) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(6 * n_cols, 5 * n_rows),
        sharey=True,
        squeeze=False,
    )

    axes = axes.ravel()

    for ax, trial in zip(axes, trials):

        subset = (
            material_data[
                material_data["Trial"].astype(int) == trial
            ]
            .sort_values("Time from elbow")
            .copy()
        )

        time = subset["Time from elbow"].to_numpy(dtype=float)
        primary = subset["Primary aligned"].to_numpy(dtype=float)
        secondary = subset["Secondary aligned"].to_numpy(dtype=float)

        # Need enough points for second derivatives
        if len(time) < 5:
            ax.axis("off")
            continue

        # Smooth before differentiating
        primary_smooth = smooth_signal(
            primary,
            window=11,
            polyorder=2,
        )

        secondary_smooth = smooth_signal(
            secondary,
            window=11,
            polyorder=2,
        )

        # First derivatives
        primary_dsignal_dt = np.gradient(
            primary_smooth,
            time,
        )

        secondary_dsignal_dt = np.gradient(
            secondary_smooth,
            time,
        )

        # Second derivatives
        primary_d2signal_dt2 = np.gradient(
            primary_dsignal_dt,
            time,
        )

        secondary_d2signal_dt2 = np.gradient(
            secondary_dsignal_dt,
            time,
        )

        # Difference of second-order derivatives
        second_derivative_difference = (
            secondary_d2signal_dt2
            - primary_d2signal_dt2
        )

        ax.plot(
            time,
            second_derivative_difference,
            linewidth=2,
            label="Secondary d²/dt² - Primary d²/dt²",
        )

        ax.axhline(
            0,
            linestyle=":",
            linewidth=1,
        )

        ax.set(
            xlabel="Time from elbow (s)",
            ylabel="Difference in d²(signal)/dt² (A/s²)",
            title=f"{material.replace('_', ' ').title()} - Trial {trial}",
        )

        ax.grid(
            linestyle=":",
            alpha=0.5,
        )

        ax.legend(fontsize=8)

    for ax in axes[len(trials):]:
        ax.axis("off")

    fig.suptitle(
        f"{material.replace('_', ' ').title()} - Difference of Second-Order Derivatives",
        fontsize=15,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()